<a href="https://colab.research.google.com/github/mohamedsylla1-ai/APLLI/blob/main/deeplearningessai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from re import X
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
df = pd.read_csv('/content/images_metadata.csv')
print(df.head())
print(df['classe'].value_counts())
print(df['dataset'].value_counts())
print(df.isnull().sum())
print(df.dtypes)

# The line 'print = ( les données )' caused a SyntaxError and has been removed or commented out.
# If you intended to print a string, please use: print('les données')

X = df[['largeur', 'hauteur', 'luminosite_moy', 'contraste', 'saturation']].values
le = LabelEncoder()
y = le.fit_transform(df['classe'])

train_mask = df['dataset'] == 'train'
val_mask = df['dataset'] == 'validation'
test_mask = df['dataset'] == 'test'

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask] # Corrected: apply val_mask to y
X_test, y_test = X[test_mask], y[test_mask]

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

print ('Notre premier reseau de neurones' )
X_train_norm = X_train / 255.0
X_val_norm = X_val / 255.0
X_test_norm = X_test / 255.0 # Corrected: division by 255.0 for normalization
model = keras.Sequential([
    layers.Input(shape=(5,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'), # Added missing layer from instructions
    layers.Dropout(0.3), # Added missing layer from instructions
    layers.Dense(1, activation='sigmoid') # Added missing output layer from instructions
])

model.summary()

# Compiler le modèle
model.compile(
    optimizer='adam',  # Algorithme d'optimisation
    loss='binary_crossentropy',  # Pour classification binaire
    metrics=['accuracy']  # Métrique à suivre
)

# Entraîner le modèle
history = model.fit(
    X_train_norm, y_train,
    validation_data=(X_val_norm, y_val),
    epochs=50,  # Nombre de passages sur les données
    batch_size=4,  # Taille des mini-batches
    verbose=1
)

# Visualiser l'apprentissage
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Courbes de perte')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Courbes de précision')

plt.tight_layout()
plt.show()

from sklearn.metrics import classification_report, confusion_matrix

# Évaluation sur le test set
test_loss, test_acc = model.evaluate(X_test_norm, y_test)
print(f"\nTest Accuracy: {test_acc:.2%}")

# Prédictions
y_pred_proba = model.predict(X_test_norm)
y_pred = (y_pred_proba > 0.5).astype(int)

# Rapport détaillé
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Chat', 'Chien']))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
print("\nMatrice de Confusion:")
print(cm)

# Exemples de prédictions
print("\nExemples de prédictions:")
for i in range(min(5, len(X_test))):
    vraie_classe = 'Chat' if y_test[i] == 0 else 'Chien'
    pred_classe = 'Chat' if y_pred[i] == 0 else 'Chien'
    confiance = y_pred_proba[i][0] if y_pred[i] == 1 else 1 - y_pred_proba[i][0]
    print(f"Image {i+1}: Vrai={vraie_classe}, Prédit={pred_classe}, Confiance={confiance:.1%}")
# Expérimentations à tester :

# 1. Ajouter plus de couches
model2 = keras.Sequential([
    layers.Input(shape=(5,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

# Compile model2
model2.compile(
    optimizer='adam',  # Algorithme d'optimisation
    loss='binary_crossentropy',  # Pour classification binaire
    metrics=['accuracy']  # Métrique à suivre
)

# 2. Changer le taux de dropout
# 3. Modifier le learning rate
# 4. Essayer d'autres optimizers : SGD, RMSprop
# 5. Implémenter Early Stopping

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,  # Arrête si pas d'amélioration après 10 epochs
    restore_best_weights=True
)

# Réentraîner avec early stopping
history2 = model2.fit(
    X_train_norm, y_train,
    validation_data=(X_val_norm, y_val),
    epochs=100,
    batch_size=4,
    callbacks=[early_stop],
    verbose=1
)

voici un récapitulatif de ce qui a été fait jusqu'à présent:

Préparation des données:

Initialement, le fichier images_metadata.csv était manquant. J'ai donc créé un fichier factice avec des données synthétiques (caractéristiques comme la largeur, la hauteur, la luminosité, le contraste, la saturation) pour simuler le jeu de données attendu.
Ces données ont été chargées dans un DataFrame pandas.
Les colonnes classe (chat/chien) ont été encodées numériquement (0 et 1).
Les données ont été divisées en ensembles d'entraînement, de validation et de test en utilisant les masques train_mask, val_mask et test_mask basés sur la colonne dataset. J'ai corrigé une erreur où y_val n'était pas correctement masqué.
Les caractéristiques (X_train, X_val, X_test) ont été normalisées en les divisant par 255.0.
Création et entraînement du premier modèle de réseau de neurones (model):

Un modèle keras.Sequential a été construit avec une couche d'entrée (5 caractéristiques), deux couches denses cachées de 64 et 32 neurones (avec activation 'relu' et des couches 'Dropout' pour la régularisation), et une couche de sortie de 1 neurone avec activation 'sigmoid' pour la classification binaire.
Le modèle a été compilé avec l'optimiseur 'adam', la fonction de perte 'binary_crossentropy' et la métrique 'accuracy'.
Il a été entraîné sur les données d'entraînement pour 50 époques, en utilisant l'ensemble de validation pour surveiller les performances.
Évaluation du premier modèle:

Les courbes de perte et de précision ont été tracées pour les ensembles d'entraînement et de validation, montrant un surapprentissage sévère (la perte d'entraînement diminuait mais la perte de validation augmentait).
L'évaluation sur l'ensemble de test a montré une très faible précision (environ 16.67%).
Le rapport de classification et la matrice de confusion ont révélé que le modèle prédisait presque toujours la classe 'Chat', ne parvenant pas à identifier correctement les 'Chiens'.
Création et entraînement du deuxième modèle de réseau de neurones (model2):

Un deuxième modèle a été défini avec une architecture plus complexe (plus de neurones et des taux de dropout légèrement différents) pour explorer des améliorations potentielles.
J'ai corrigé une erreur où ce deuxième modèle n'était pas compilé avant l'entraînement.
Ce modèle a également été entraîné avec l'optimiseur 'adam', la fonction de perte 'binary_crossentropy', et la métrique 'accuracy', mais avec l'EarlyStopping pour arrêter l'entraînement si la performance sur l'ensemble de validation n'améliorait pas après 10 époques.
Le model2 a également montré des signes de surapprentissage et l'entraînement a été arrêté prématurément par EarlyStopping.
En résumé, bien que les étapes techniques de construction et d'entraînement des modèles aient été exécutées, les performances des deux modèles sont très faibles en raison du jeu de données très petit et synthétique, qui ne permet pas aux modèles de deep learning d'apprendre des motifs significatifs.

# EXERCICE 11 : DEEP LEARNING - CLASSIFICATION D'IMAGES 🖼️

## 🎯 OBJECTIF
**Votre première** classification d'images avec un réseau de neurones convolutif (CNN).

## 📁 FICHIERS FOURNIS
- `images_metadata.csv` : Métadonnées de 24 images (chats vs chiens)

## 🧠 INTRODUCTION AU DEEP LEARNING

### Qu'est-ce qu'un CNN ?
Un **Convolutional Neural Network** apprend automatiquement à reconnaître des patterns visuels :
- **Couches convolutives** : Détectent les formes (bords, textures, motifs)
- **Couches de pooling** : Réduisent la taille tout en gardant l'essentiel
- **Couches denses** : Prennent la décision finale

### Pourquoi c'est différent du ML classique ?
- ML : Vous créez les features manuellement
- DL : Le réseau apprend les features automatiquement

## 📋 TÂCHES ÉTAPE PAR ÉTAPE

### ÉTAPE 1 : Comprendre les données (15 min)

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Charger les métadonnées
df = pd.read_csv('images_metadata.csv')
print(df.head())
print(df['classe'].value_counts())
print(df['dataset'].value_counts())

# Note : Nous simulons des images avec des données synthétiques
# Dans un vrai projet, vous chargeriez les vraies images
```

**Questions :**
- Combien d'images par classe ?
- Comment sont réparties train/validation/test ?

### ÉTAPE 2 : Préparer les données (20 min)

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Pour cet exercice, nous allons simuler des images avec les features disponibles
# En réalité, vous utiliseriez PIL ou OpenCV pour charger les images

# Créer des features "image-like" basées sur les métadonnées
X = df[['largeur', 'hauteur', 'luminosite_moy', 'contraste', 'saturation']].values

# Encoder les labels
le = LabelEncoder()
y = le.fit_transform(df['classe'])  # 0=chat, 1=chien

# Split selon le dataset
train_mask = df['dataset'] == 'train'
val_mask = df['dataset'] == 'validation'
test_mask = df['dataset'] == 'test'

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
```

### ÉTAPE 3 : Créer votre premier réseau de neurones (30 min)

```python
from tensorflow import keras
from tensorflow.keras import layers

# IMPORTANT : Normaliser les données
X_train_norm = X_train / 255.0  # Simulation de normalisation
X_val_norm = X_val / 255.0
X_test_norm = X_test / 255.0

# Créer le modèle séquentiel
model = keras.Sequential([
    # Couche d'entrée
    layers.Input(shape=(5,)),  # 5 features
    
    # Couches cachées
    layers.Dense(64, activation='relu'),  # 64 neurones
    layers.Dropout(0.3),  # Évite l'overfitting
    
    layers.Dense(32, activation='relu'),  # 32 neurones
    layers.Dropout(0.3),
    
    # Couche de sortie
    layers.Dense(1, activation='sigmoid')  # 1 neurone pour binaire
])

# Afficher l'architecture
model.summary()
```

**Comprendre l'architecture :**
- `Dense(64)` : 64 neurones connectés à tous les précédents
- `relu` : Fonction d'activation (ajoute de la non-linéarité)
- `Dropout` : Désactive aléatoirement 30% des neurones (anti-overfitting)
- `sigmoid` : Pour classification binaire (sortie entre 0 et 1)

### ÉTAPE 4 : Compiler et entraîner (30 min)

```python
# Compiler le modèle
model.compile(
    optimizer='adam',  # Algorithme d'optimisation
    loss='binary_crossentropy',  # Pour classification binaire
    metrics=['accuracy']  # Métrique à suivre
)

# Entraîner le modèle
history = model.fit(
    X_train_norm, y_train,
    validation_data=(X_val_norm, y_val),
    epochs=50,  # Nombre de passages sur les données
    batch_size=4,  # Taille des mini-batches
    verbose=1
)

# Visualiser l'apprentissage
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Courbes de perte')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Courbes de précision')

plt.tight_layout()
plt.show()
```

**Interpréter les courbes :**
- Loss qui diminue = Le modèle apprend ✅
- Val_loss qui augmente après un moment = Overfitting ⚠️
- Accuracy qui augmente = Bonnes prédictions ✅

### ÉTAPE 5 : Évaluer et prédire (25 min)

```python
from sklearn.metrics import classification_report, confusion_matrix

# Évaluation sur le test set
test_loss, test_acc = model.evaluate(X_test_norm, y_test)
print(f"\nTest Accuracy: {test_acc:.2%}")

# Prédictions
y_pred_proba = model.predict(X_test_norm)
y_pred = (y_pred_proba > 0.5).astype(int)

# Rapport détaillé
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Chat', 'Chien']))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
print("\nMatrice de Confusion:")
print(cm)

# Exemples de prédictions
print("\nExemples de prédictions:")
for i in range(min(5, len(X_test))):
    vraie_classe = 'Chat' if y_test[i] == 0 else 'Chien'
    pred_classe = 'Chat' if y_pred[i] == 0 else 'Chien'
    confiance = y_pred_proba[i][0] if y_pred[i] == 1 else 1 - y_pred_proba[i][0]
    print(f"Image {i+1}: Vrai={vraie_classe}, Prédit={pred_classe}, Confiance={confiance:.1%}")
```

### ÉTAPE 6 : Améliorer le modèle (optionnel, 30 min)

```python
# Expérimentations à tester :

# 1. Ajouter plus de couches
model2 = keras.Sequential([
    layers.Input(shape=(5,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

# 2. Changer le taux de dropout
# 3. Modifier le learning rate
# 4. Essayer d'autres optimizers : SGD, RMSprop
# 5. Implémenter Early Stopping

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,  # Arrête si pas d'amélioration après 10 epochs
    restore_best_weights=True
)

# Réentraîner avec early stopping
history2 = model2.fit(
    X_train_norm, y_train,
    validation_data=(X_val_norm, y_val),
    epochs=100,
    batch_size=4,
    callbacks=[early_stop],
    verbose=1
)
```

## 💡 CONCEPTS CLÉS À RETENIR

### 1. **Architecture des réseaux**
- Plus de neurones = Plus de capacité (mais risque d'overfitting)
- Dropout = Régularisation pour éviter l'overfitting

### 2. **Hyperparamètres**
- `epochs` : Nombre de fois qu'on voit toutes les données
- `batch_size` : Nombre d'exemples par mise à jour
- `learning_rate` : Vitesse d'apprentissage

### 3. **Overfitting vs Underfitting**
- **Overfitting** : Modèle trop complexe → mémorise au lieu d'apprendre
- **Underfitting** : Modèle trop simple → n'apprend pas assez

### 4. **Fonctions d'activation**
- `relu` : f(x) = max(0, x) → Pour couches cachées
- `sigmoid` : f(x) = 1/(1+e^-x) → Pour classification binaire
- `softmax` : Pour classification multi-classes

## ✅ CRITÈRES DE RÉUSSITE
- [ ] Accuracy > 70% sur le test set
- [ ] Comprendre l'architecture du modèle
- [ ] Interpréter les courbes d'apprentissage
- [ ] Identifier l'overfitting si présent
- [ ] Sauvegarder le modèle

## 🚀 POUR ALLER PLUS LOIN

### Avec de vraies images
```python
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation (rotation, zoom, flip...)
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

# Charger des images depuis des dossiers
train_generator = datagen.flow_from_directory(
    'data/train/',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)
```

### Transfer Learning
```python
# Utiliser un modèle pré-entraîné (VGG16, ResNet, EfficientNet)
base_model = keras.applications.VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Geler les couches pré-entraînées
base_model.trainable = False

# Ajouter vos propres couches
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])
```

## 📊 LIVRABLES
1. Script Python avec le modèle entraîné
2. Graphiques : courbes loss/accuracy
3. Rapport de classification
4. Modèle sauvegardé (.h5 ou .keras)
5. Document : architecture choisie et pourquoi

**Durée : 2h30**
**Niveau : ⭐⭐ (Débutant DL)**

## 🎓 RESSOURCES
- TensorFlow/Keras documentation
- CS231n (Stanford) - Cours gratuit sur les CNN
- Deep Learning Book (Goodfellow et al.)